In [0]:
VOLUME_PATH = "/Volumes/ifood_case/default/landing/ny_taxi_trip/yellow/"
CATALOG_NAME = "ifood_case"
SCHEMA_NAME = "raw"
TABLE_NAME = "ny_taxi_trip_yellow"

In [0]:
from functools import reduce
from pyspark.sql import functions as F

landing_paths = [f"{VOLUME_PATH}/{y.name}{m.name}"
         for y in dbutils.fs.ls(VOLUME_PATH)
         for m in dbutils.fs.ls(f"{VOLUME_PATH}/{y.name}")]

def read_as_string(path):
    df = ( spark.read
          .option("basePath", VOLUME_PATH)
          .parquet(path)
    )
    return ( df.select([F.col(c).cast("string").alias(c) for c in df.columns])
                .withColumn("month", F.lpad(F.col("month"), 2, "0"))
    )

final_df = reduce(lambda all_dfs, next_df: all_dfs.unionByName(next_df, allowMissingColumns=True),
            [read_as_string(landing_path) for landing_path in landing_paths])

In [0]:
final_df.write.format("delta").mode("overwrite") \
  .partitionBy("year", "month") \
  .option("partitionOverwriteMode", "dynamic") \
  .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")

In [0]:
[dbutils.fs.rm(landing_path, recurse=True) for landing_path in landing_paths]